# LinguoAI für Google Colab

Diese Version installiert und startet LinguoAI in **einer einzigen Zelle**. Dadurch tritt `ModuleNotFoundError: No module named 'linguoai'` nicht mehr auf, wenn frühere Setup-Zellen ausgelassen wurden.

Enthalten sind Mehrfach-Upload, sequenzielle Batch-Verarbeitung, Gemini-Pausen, Wiederholungsversuche und automatischer Wechsel zu Google Translate bei Gemini-Ausfall.


In [ ]:
# @title 1. LinguoAI installieren und starten
# Diese einzige Zelle erledigt Setup, Installation, Importprüfung und Start.
import importlib
import importlib.util
import os
import shutil
import stat
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

PROJECT_SOURCE = "upload_zip"  # @param ["upload_zip", "download_zip"]
SOURCE_ZIP_URL = "https://linguoai.de/downloads/LinguoAI-2.0.0-source.zip"  # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = False  # @param {type:"boolean"}
CACHE_MODELS_IN_DRIVE = False  # @param {type:"boolean"}
LOCAL_TTS_EXTRA = "none"  # @param ["none", "piper", "chatterbox"]

print("[1/5] Colab-Umgebung wird vorbereitet …")
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    print("Bitte die Google-Drive-Freigabe im Browser bestätigen.")
    drive.mount("/content/drive", force_remount=False)

if CACHE_MODELS_IN_DRIVE and MOUNT_GOOGLE_DRIVE:
    model_cache = Path("/content/drive/MyDrive/LinguoAI/model-cache")
else:
    model_cache = Path("/content/linguoai-model-cache")
model_cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(model_cache)
os.environ["LINGUOAI_MODEL_CACHE"] = str(model_cache)
print("Modellcache:", model_cache)

project_area = Path("/content/linguoai-project")
project_area.mkdir(parents=True, exist_ok=True)


def safe_extract_zip(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        if len(entries) > 10000 or sum(item.file_size for item in entries) > 750 * 1024**2:
            raise ValueError("Das Projekt-ZIP ist ungewöhnlich groß.")
        for info in entries:
            member = Path(info.filename)
            target = (root / member).resolve()
            mode = (info.external_attr >> 16) & 0o170000
            if member.is_absolute() or ".." in member.parts or not target.is_relative_to(root):
                raise ValueError(f"Unsicherer ZIP-Pfad: {info.filename}")
            if mode == stat.S_IFLNK:
                raise ValueError(f"Symlink im ZIP ist nicht erlaubt: {info.filename}")
        archive.extractall(root)


print("[2/5] LinguoAI-Quellcode wird geladen …")
extraction = project_area / "source"
if extraction.exists():
    shutil.rmtree(extraction)

archive_path = project_area / "LinguoAI-source.zip"
if PROJECT_SOURCE == "download_zip":
    if not SOURCE_ZIP_URL.startswith("https://"):
        raise ValueError("SOURCE_ZIP_URL muss mit https:// beginnen.")
    request = urllib.request.Request(
        SOURCE_ZIP_URL,
        headers={"User-Agent": "LinguoAI-Colab"},
    )
    with urllib.request.urlopen(request, timeout=180) as response, archive_path.open("wb") as out:
        while chunk := response.read(1024 * 1024):
            out.write(chunk)
else:
    from google.colab import files
    print("Bitte jetzt LinguoAI-2.0.0-source.zip auswählen.")
    uploaded = files.upload()
    archives = [(name, data) for name, data in uploaded.items() if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Bitte genau ein LinguoAI-Quellcode-ZIP hochladen.")
    archive_path.write_bytes(archives[0][1])

safe_extract_zip(archive_path, extraction)
candidates = [
    path.parent
    for path in extraction.rglob("pyproject.toml")
    if (path.parent / "linguoai" / "colab_ui.py").is_file()
]
if len(candidates) != 1:
    raise RuntimeError(f"LinguoAI-Projektwurzel nicht eindeutig gefunden: {candidates}")
project_root = candidates[0]
print("Projektwurzel:", project_root)

print("[3/5] System- und Python-Abhängigkeiten werden installiert …")
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)

extras = ["colab"]
if LOCAL_TTS_EXTRA == "piper":
    extras.append("piper")
elif LOCAL_TTS_EXTRA == "chatterbox":
    extras.append("voice-clone")
install_target = f"{project_root}[{','.join(extras)}]"
print("pip install -e", install_target)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "-e", install_target],
    check=True,
)

# Editable-Installationen werden in laufenden Notebooks nicht immer sofort gefunden.
# Der Projektpfad wird deshalb zusätzlich explizit eingetragen und alte Imports entfernt.
for module_name in list(sys.modules):
    if module_name == "linguoai" or module_name.startswith("linguoai."):
        del sys.modules[module_name]
sys.path.insert(0, str(project_root))
importlib.invalidate_caches()

print("[4/5] LinguoAI-Import wird geprüft …")
from linguoai import __version__
from linguoai.colab_ui import launch_colab

if __version__ != "2.2.0":
    raise RuntimeError(f"Falsche LinguoAI-Version geladen: {__version__}")
print("LinguoAI", __version__, "wurde erfolgreich geladen.")

print("[5/5] Oberfläche wird gestartet …")
print("Hinweis: Die Zelle bleibt absichtlich aktiv, solange die Gradio-Oberfläche läuft.")
print("Öffne gleich den angezeigten öffentlichen Link und melde dich mit den Zugangsdaten an.")
launch_colab()
